## Prediction of Customer Churn Pbobability

**Version 1.0**
The notebook in fourth entry to competition I started with Basics to start with using Random Fore7r, before I test pipelione and ensemble

**Version 2.0**
Random Forests tend to struggle with this because they average predictions from many trees, which often pushes probabilities away from 0 and 1 toward the center.
This version introduces Random Forest Calibration By applying Platt Scaling (Sigmoid calibration), turns a mathematical output into a reliable financial metric for churn prevention strategies.

**Version 3.0**
Introduced GridSearchCV: It no longer guesses the number of trees; it tests 50, 100, 150, 200, 250 and 300 to find the most accurate estimator for specific data.

**Version 4.0**
Introduced notebook pipeline using Balanced Random Forest (from imblearn) designed to:
- Handle churn imbalance properly
- Increase churn probability sensitivity
- Optimize for ROC-AUC
- Improve recall for churners
- Provide calibrated probabilities
- Use stratified CV
- Automatically tune threshold

**Version 5.0***
Added multi-seed ensemble combining XGBoost + Random Forest, optimized for churn imbalance and probability separation.
- XGBoost with scale_pos_weight
- RandomForest with class_weight='balanced'
- 3 seeds
- 5-fold stratified CV
- Rank normalization blending
- Weight optimization
- Threshold tuning for recall

In [1]:
# ============================================================
# 1. Imports
# ============================================================

import numpy as np
import pandas as pd
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import rankdata

import xgboost as xgb

SEEDS = [42, 333, 777]
N_FOLDS = 5
TARGET = "Churn"

# Set visual style
sns.set(style="whitegrid")


In [2]:
# =============================================================================
# 2. Data Loading 
# =============================================================================
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/test.csv")

print("Completed Loading Data ...")
print("Train shape:", train.shape)
print("Test shape :", test.shape)

test_ids = test["id"].copy()

# Descriptive Statistics
stats = train.describe()

print("\nDescriptive Stats:\n", stats)


Completed Loading Data ...
Train shape: (594194, 21)
Test shape : (254655, 20)

Descriptive Stats:
                   id  SeniorCitizen         tenure  MonthlyCharges  \
count  594194.000000  594194.000000  594194.000000   594194.000000   
mean   297096.500000       0.114102      36.577258       65.866223   
std    171529.177262       0.317936      25.061922       31.067444   
min         0.000000       0.000000       1.000000       18.250000   
25%    148548.250000       0.000000      12.000000       29.900000   
50%    297096.500000       0.000000      35.000000       74.100000   
75%    445644.750000       0.000000      62.000000       90.800000   
max    594193.000000       1.000000      72.000000      118.750000   

        TotalCharges  
count  594194.000000  
mean     2494.377057  
std      2353.916710  
min        18.800000  
25%       639.650000  
50%      1433.650000  
75%      4263.800000  
max      8684.800000  


In [3]:
# ============================================================
# 3. Cleaning & Encoding
# ============================================================

for df in [train, test]:
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
    df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)

cat_cols = train.select_dtypes(include="object").columns.tolist()
cat_cols.remove(TARGET)

le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col].astype(str))
    test[col]  = le.transform(test[col].astype(str))
    le_dict[col] = le

train[TARGET] = train[TARGET].map({"Yes":1, "No":0})

X = train.drop(["id", TARGET], axis=1)
y = train[TARGET]
X_test = test.drop(["id"], axis=1)

print("Churn rate:", y.mean())

neg = (y == 0).sum()
pos = (y == 1).sum()
scale_pos_weight = neg / pos

print("scale_pos_weight:", scale_pos_weight)

Churn rate: 0.225207592133209
scale_pos_weight: 3.440347638939746


In [4]:
# ============================================================
# 4. Multi-Seed Training - XGB + RF
# ============================================================

all_oof_xgb = []
all_oof_rf = []

all_test_xgb = []
all_test_rf = []

for seed in SEEDS:

    print(f"\n{'='*60}")
    print(f"Training Seed {seed}")
    print(f"{'='*60}")

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)

    oof_xgb = np.zeros(len(X))
    oof_rf = np.zeros(len(X))

    test_xgb = np.zeros(len(X_test))
    test_rf = np.zeros(len(X_test))

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):

        print(f"\nSeed {seed} - Fold {fold+1}")

        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        # ---------------- XGBoost ----------------
        xgb_model = xgb.XGBClassifier(
            n_estimators=20000,
            learning_rate=0.03,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=5,
            gamma=0.1,
            reg_alpha=0.1,
            reg_lambda=1.0,
            scale_pos_weight=scale_pos_weight,
            random_state=seed,
            tree_method="hist",
            objective="binary:logistic",
            eval_metric="auc",
            n_jobs=-1
        )

        xgb_model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            verbose=False
        )

        oof_xgb[va_idx] = xgb_model.predict_proba(X_va)[:,1]
        test_xgb += xgb_model.predict_proba(X_test)[:,1] / N_FOLDS

        # ---------------- Random Forest ----------------
        rf_model = RandomForestClassifier(
            n_estimators=600,
            max_depth=None,
            min_samples_leaf=3,
            class_weight="balanced",
            random_state=seed,
            n_jobs=-1
        )

        rf_model.fit(X_tr, y_tr)

        oof_rf[va_idx] = rf_model.predict_proba(X_va)[:,1]
        test_rf += rf_model.predict_proba(X_test)[:,1] / N_FOLDS

        print("Fold AUC XGB:", roc_auc_score(y_va, oof_xgb[va_idx]))
        print("Fold AUC RF :", roc_auc_score(y_va, oof_rf[va_idx]))

    print("Seed OOF AUC XGB:", roc_auc_score(y, oof_xgb))
    print("Seed OOF AUC RF :", roc_auc_score(y, oof_rf))

    all_oof_xgb.append(oof_xgb)
    all_oof_rf.append(oof_rf)

    all_test_xgb.append(test_xgb)
    all_test_rf.append(test_rf)


Training Seed 42

Seed 42 - Fold 1
Fold AUC XGB: 0.9093760022045475
Fold AUC RF : 0.9098553413919415

Seed 42 - Fold 2
Fold AUC XGB: 0.9107961249272809
Fold AUC RF : 0.9113557823077674

Seed 42 - Fold 3
Fold AUC XGB: 0.9098703004465414
Fold AUC RF : 0.910157686864882

Seed 42 - Fold 4
Fold AUC XGB: 0.9109401760819817
Fold AUC RF : 0.9113279289215055

Seed 42 - Fold 5
Fold AUC XGB: 0.908316574823085
Fold AUC RF : 0.9088208917915039
Seed OOF AUC XGB: 0.9098553474291926
Seed OOF AUC RF : 0.9102977784583468

Training Seed 333

Seed 333 - Fold 1
Fold AUC XGB: 0.9115200459933692
Fold AUC RF : 0.9117080970284823

Seed 333 - Fold 2
Fold AUC XGB: 0.9095763558656929
Fold AUC RF : 0.9094618334788318

Seed 333 - Fold 3
Fold AUC XGB: 0.9087443424495433
Fold AUC RF : 0.90952490677558

Seed 333 - Fold 4
Fold AUC XGB: 0.9092469173641649
Fold AUC RF : 0.9105911923380285

Seed 333 - Fold 5
Fold AUC XGB: 0.9101063663286131
Fold AUC RF : 0.9107727161991682
Seed OOF AUC XGB: 0.9098289473561779
Seed OOF AU

In [5]:
# =============================================================================
# 5. Average across the SEED
# =============================================================================

oof_xgb_final = np.mean(all_oof_xgb, axis=0)
oof_rf_final  = np.mean(all_oof_rf, axis=0)

test_xgb_final = np.mean(all_test_xgb, axis=0)
test_rf_final  = np.mean(all_test_rf, axis=0)

print("\nMulti-Seed OOF AUC XGB:", roc_auc_score(y, oof_xgb_final))
print("Multi-Seed OOF AUC RF :", roc_auc_score(y, oof_rf_final))




Multi-Seed OOF AUC XGB: 0.9112700578134114
Multi-Seed OOF AUC RF : 0.9111673655289771


In [6]:
# =============================================================================
# 6. Rank Optimization
# =============================================================================

def rank_norm(x):
    return rankdata(x) / len(x)

oof_xgb_r = rank_norm(oof_xgb_final)
oof_rf_r  = rank_norm(oof_rf_final)

test_xgb_r = rank_norm(test_xgb_final)
test_rf_r  = rank_norm(test_rf_final)

In [7]:
# =============================================================================
# 7. Optimize Ensemble Weight
# =============================================================================
best_auc = 0
best_w = 0.5

for w in np.arange(0.0, 1.01, 0.05):
    blend = w * oof_xgb_r + (1-w) * oof_rf_r
    auc = roc_auc_score(y, blend)
    if auc > best_auc:
        best_auc = auc
        best_w = w

print("\nOptimal Weight for XGB:", best_w)
print("Optimal Weight for RF :", 1-best_w)
print("Ensemble OOF AUC:", best_auc)


Optimal Weight for XGB: 0.5
Optimal Weight for RF : 0.5
Ensemble OOF AUC: 0.9133960675703221


In [8]:
# =============================================================================
# 8. Optimize Ensemble Weight
# =============================================================================

blend_oof = best_w * oof_xgb_r + (1-best_w) * oof_rf_r

thresholds = np.arange(0.2, 0.6, 0.02)
best_recall = 0
best_t = 0.5

for t in thresholds:
    preds = (blend_oof >= t).astype(int)
    recall = classification_report(y, preds, output_dict=True)["1"]["recall"]
    if recall > best_recall:
        best_recall = recall
        best_t = t

print("\nBest Recall:", best_recall)
print("Optimal Threshold:", best_t)


Best Recall: 0.9977431865906424
Optimal Threshold: 0.2


In [9]:
# =============================================================================
# 9. Final Predictions
# =============================================================================
final_test = best_w * test_xgb_r + (1-best_w) * test_rf_r

submission = pd.DataFrame({
    "id": test_ids,
    "Churn_Probability": np.round(final_test, 5)
})

submission.to_csv("submission.csv", index=False)
submission.head(20)



,id,Churn_Probability
0,594194,0.51815
1,594195,0.02608
2,594196,0.56873
3,594197,0.14701
4,594198,0.80333
5,594199,0.64361
6,594200,0.98592
7,594201,0.27283
8,594202,0.40920
9,594203,0.73229
